In [21]:
#H2O calculation and result

%pip install -q qiskit qiskit-nature qiskit-algorithms qiskit-ibm-runtime pyscf scipy

Note: you may need to restart the kernel to use updated packages.


In [22]:
import qiskit
import qiskit_nature
import qiskit_algorithms
import qiskit_ibm_runtime
import pyscf

print("All packages imported successfully!")
print("Qiskit:", qiskit.__version__)
print("Qiskit Nature:", qiskit_nature.__version__)
print("IBM Runtime:", qiskit_ibm_runtime.__version__)

All packages imported successfully!
Qiskit: 2.5.0
Qiskit Nature: 0.8.0
IBM Runtime: 0.41.1


In [23]:
from getpass import getpass

ibm_token = getpass("Enter your IBM Quantum API key: ")

print("API key received!")

API key received!


In [24]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=ibm_token
)

print("Successfully connected to IBM Quantum!")

qiskit_runtime_service._resolve_cloud_instances:WARNING:2026-11-07 14:13:45,017: Default instance not set. Searching all available instances.


Successfully connected to IBM Quantum!


In [25]:
backends = service.backends()

print("Available quantum computers:")

for backend in backends:
    print(
        "Name:", backend.name,
        "| Qubits:", backend.num_qubits,
        "| Status:", backend.status().status_msg
    )

Available quantum computers:
Name: ibm_fez | Qubits: 156 | Status: active
Name: ibm_kingston | Qubits: 156 | Status: active
Name: ibm_marrakesh | Qubits: 156 | Status: active


In [26]:
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=14
)

print("Selected processor:", backend.name)
print("Physical qubits:", backend.num_qubits)
print("Jobs waiting:", backend.status().pending_jobs)

Selected processor: ibm_kingston
Physical qubits: 156
Jobs waiting: 9


In [27]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)

qc.h(0)
qc.cx(0, 1)

qc.measure([0, 1], [0, 1])

print(qc.draw())

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [28]:
from qiskit import transpile
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Adapt the circuit to the selected IBM processor
isa_qc = transpile(qc, backend=backend, optimization_level=1)

# Create the hardware sampler
sampler = Sampler(mode=backend)

# Submit the circuit
job = sampler.run([isa_qc], shots=1024)

print("Circuit submitted successfully!")
print("Processor:", backend.name)
print("Job ID:", job.job_id())

Circuit submitted successfully!
Processor: ibm_kingston
Job ID: dbnj4boou4as73b4m0r0


In [30]:
print("Job status:", job.status())

Job status: QUEUED


In [14]:
result = job.result()

counts = result[0].data.c.get_counts()

print("Measurement results:")
print(counts)

KeyboardInterrupt: 

In [11]:
from pyscf import gto, scf
from qiskit_nature.second_q.drivers import PySCFDriver

print("PySCF and Qiskit Nature are working!")

PySCF and Qiskit Nature are working!


In [12]:
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver

driver = PySCFDriver(
    atom=[
        "O  0.000  0.000  0.000",
        "H  0.757  0.000  0.586",
        "H -0.757  0.000  0.586"
    ],
    unit=DistanceUnit.ANGSTROM,
    basis="sto3g",
    charge=0,
    spin=0
)

problem = driver.run()

print("Water molecule successfully created!")
print("Number of electrons:", problem.num_particles)
print("Number of spatial orbitals:", problem.num_spatial_orbitals)

Water molecule successfully created!
Number of electrons: (5, 5)
Number of spatial orbitals: 7


In [13]:
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

transformer = ActiveSpaceTransformer(
    num_electrons=4,
    num_spatial_orbitals=4
)

active_problem = transformer.transform(problem)

print("Active-space model created!")
print("Electrons:", active_problem.num_particles)
print("Spatial orbitals:", active_problem.num_spatial_orbitals)

Active-space model created!
Electrons: (2, 2)
Spatial orbitals: 4


In [14]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()

fermionic_hamiltonian = active_problem.hamiltonian.second_q_op()

qubit_hamiltonian = mapper.map(fermionic_hamiltonian)

print("Qubit Hamiltonian created!")
print("Number of qubits:", qubit_hamiltonian.num_qubits)
print("Number of Hamiltonian terms:", len(qubit_hamiltonian))

Qubit Hamiltonian created!
Number of qubits: 8
Number of Hamiltonian terms: 105


In [15]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

# Create the Hartree-Fock starting state
hf_state = HartreeFock(
    num_spatial_orbitals=active_problem.num_spatial_orbitals,
    num_particles=active_problem.num_particles,
    qubit_mapper=mapper
)

# Create the UCCSD quantum circuit
ansatz = UCCSD(
    num_spatial_orbitals=active_problem.num_spatial_orbitals,
    num_particles=active_problem.num_particles,
    qubit_mapper=mapper,
    initial_state=hf_state
)

print("UCCSD circuit created!")
print("Number of qubits:", ansatz.num_qubits)
print("Number of parameters:", ansatz.num_parameters)
print("Circuit depth:", ansatz.decompose().depth())

UCCSD circuit created!
Number of qubits: 8
Number of parameters: 26
Circuit depth: 27


In [16]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)

print("IBM quantum estimator created!")
print("Processor:", backend.name)

IBM quantum estimator created!
Processor: ibm_fez


In [17]:
from qiskit.transpiler.preset_passmanagers import (
    generate_preset_pass_manager
)

# Create a hardware-specific transpiler
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1
)

# Transpile the UCCSD circuit
isa_ansatz = pm.run(ansatz)

# Adjust the Hamiltonian to match the physical qubit layout
isa_hamiltonian = qubit_hamiltonian.apply_layout(
    isa_ansatz.layout
)

print("Hardware transpilation successful!")
print("Logical qubits:", ansatz.num_qubits)
print("Physical circuit qubits:", isa_ansatz.num_qubits)
print("Transpiled circuit depth:", isa_ansatz.depth())
print("Number of parameters:", isa_ansatz.num_parameters)

Hardware transpilation successful!
Logical qubits: 8
Physical circuit qubits: 156
Transpiled circuit depth: 6443
Number of parameters: 26


In [18]:
print("Gate counts:")
print(isa_ansatz.count_ops())

print("\nTotal circuit operations:")
print(isa_ansatz.size())

print("\nTwo-qubit gate count:")
print(isa_ansatz.num_nonlocal_gates())

Gate counts:
OrderedDict({'rz': 3555, 'sx': 2196, 'cz': 1619, 'x': 20})

Total circuit operations:
7390

Two-qubit gate count:
1619


In [19]:
from qiskit.circuit.library import EfficientSU2

small_ansatz = EfficientSU2(
    num_qubits=8,
    su2_gates=["ry"],
    entanglement="linear",
    reps=1
)

print("Smaller ansatz created!")
print("Qubits:", small_ansatz.num_qubits)
print("Parameters:", small_ansatz.num_parameters)
print("Circuit depth:", small_ansatz.decompose().depth())

Smaller ansatz created!
Qubits: 8
Parameters: 16
Circuit depth: 9


/tmp/ipykernel_384/471105560.py:3: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  small_ansatz = EfficientSU2(


In [20]:
%pip install -q qiskit-nature pyscf

Note: you may need to restart the kernel to use updated packages.


In [21]:
import qiskit_nature
import pyscf

print("Qiskit Nature version:", qiskit_nature.__version__)
print("PySCF is working!")

Qiskit Nature version: 0.8.0
PySCF is working!


In [22]:
variables = [
    "service",
    "backend",
    "problem",
    "active_problem",
    "mapper",
    "qubit_hamiltonian",
    "small_ansatz",
    "pm",
    "estimator"
]

for name in variables:
    print(name, "exists" if name in globals() else "missing")

service exists
backend exists
problem exists
active_problem exists
mapper exists
qubit_hamiltonian exists
small_ansatz exists
pm exists
estimator exists


In [23]:
from qiskit_nature.second_q.circuit.library import HartreeFock

# Recreate Hartree-Fock state if needed
hf_state = HartreeFock(
    num_spatial_orbitals=active_problem.num_spatial_orbitals,
    num_particles=active_problem.num_particles,
    qubit_mapper=mapper
)

# Combine the starting state and smaller ansatz
small_chemistry_ansatz = hf_state.compose(small_ansatz)

# Transpile for IBM hardware
small_isa_ansatz = pm.run(small_chemistry_ansatz)

print("Smaller circuit transpiled!")
print("Circuit width:", small_isa_ansatz.num_qubits)
print("Circuit depth:", small_isa_ansatz.depth())
print("Gate counts:", small_isa_ansatz.count_ops())
print("Parameters:", small_isa_ansatz.num_parameters)

Smaller circuit transpiled!
Circuit width: 156
Circuit depth: 37
Gate counts: OrderedDict({'rz': 54, 'sx': 38, 'cz': 7})
Parameters: 16


In [24]:
# Map the molecular Hamiltonian to the physical layout
isa_hamiltonian = qubit_hamiltonian.apply_layout(
    small_isa_ansatz.layout
)

print("Hamiltonian mapped successfully!")
print("Hamiltonian qubits:", isa_hamiltonian.num_qubits)
print("Hamiltonian terms:", len(isa_hamiltonian))
print("Circuit qubits:", small_isa_ansatz.num_qubits)

Hamiltonian mapped successfully!
Hamiltonian qubits: 156
Hamiltonian terms: 105
Circuit qubits: 156


In [ ]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)
estimator.options.default_shots = 4096
estimator.options.resilience_level = 1

# Configure measurement precision
estimator.options.default_shots = 4096

print("Energy estimator configured!")
print("Processor:", backend.name)
print("Shots per measurement:", estimator.options.default_shots)

Energy estimator configured!
Processor: ibm_fez
Shots per measurement: 4096


In [26]:
import numpy as np

# Set all ansatz parameters to zero
initial_parameters = np.zeros(
    small_isa_ansatz.num_parameters
)

print("Submitting initial energy measurement...")

# Submit the molecular Hamiltonian to IBM
job = estimator.run([
    (
        small_isa_ansatz,
        isa_hamiltonian,
        initial_parameters
    )
])

print("Job submitted!")
print("Job ID:", job.job_id())

Submitting initial energy measurement...
Job submitted!
Job ID: dbncmt5t7fqc73ajc6kg


In [30]:
print("Job status:", job.status())

Job status: DONE


In [31]:
result = job.result()

energy = float(result[0].data.evs)

print("Initial molecular energy:", energy, "Hartree")

Initial molecular energy: -4.379683786726964 Hartree


In [32]:
print("Active-space Hamiltonian constants:")
print(active_problem.hamiltonian.constants)

print("\nNuclear repulsion energy:")
print(active_problem.nuclear_repulsion_energy)

print("\nNumber of electrons:")
print(active_problem.num_particles)

print("\nNumber of spatial orbitals:")
print(active_problem.num_spatial_orbitals)


Active-space Hamiltonian constants:
{'nuclear_repulsion_energy': np.float64(9.193913160623385), 'ActiveSpaceTransformer': np.float64(-78.00111916815318)}

Nuclear repulsion energy:
9.193913160623385

Number of electrons:
(2, 2)

Number of spatial orbitals:
4


In [33]:
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

# Create a smaller active-space model
small_transformer = ActiveSpaceTransformer(
    num_electrons=2,
    num_spatial_orbitals=2
)

tiny_problem = small_transformer.transform(problem)

# Map the Hamiltonian to qubits
tiny_hamiltonian = mapper.map(
    tiny_problem.hamiltonian.second_q_op()
)

# Create Hartree-Fock starting state
tiny_hf = HartreeFock(
    num_spatial_orbitals=2,
    num_particles=tiny_problem.num_particles,
    qubit_mapper=mapper
)

# Create chemistry-based UCCSD circuit
tiny_ansatz = UCCSD(
    num_spatial_orbitals=2,
    num_particles=tiny_problem.num_particles,
    qubit_mapper=mapper,
    initial_state=tiny_hf
)

print("Smaller chemistry model created!")
print("Electrons:", tiny_problem.num_particles)
print("Qubits:", tiny_ansatz.num_qubits)
print("Parameters:", tiny_ansatz.num_parameters)
print("Circuit depth:", tiny_ansatz.decompose().depth())

Smaller chemistry model created!
Electrons: (1, 1)
Qubits: 4
Parameters: 3
Circuit depth: 4


In [20]:
tiny_isa_ansatz = pm.run(tiny_ansatz)

tiny_isa_hamiltonian = tiny_hamiltonian.apply_layout(
    tiny_isa_ansatz.layout
)

print("Hardware transpilation complete!")
print("Circuit width:", tiny_isa_ansatz.num_qubits)
print("Circuit depth:", tiny_isa_ansatz.depth())
print("Gate counts:", tiny_isa_ansatz.count_ops())
print("Parameters:", tiny_isa_ansatz.num_parameters)

NameError: name 'pm' is not defined

In [19]:
from qiskit.transpiler.preset_passmanagers import (
    generate_preset_pass_manager
)

# Create a more aggressive optimization pass
pm_optimized = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3
)

# Transpile the chemistry circuit again
tiny_isa_optimized = pm_optimized.run(tiny_ansatz)

# Match the Hamiltonian to the new layout
tiny_hamiltonian_optimized = tiny_hamiltonian.apply_layout(
    tiny_isa_optimized.layout
)

print("Optimized circuit:")
print("Physical qubits:", tiny_isa_optimized.num_qubits)
print("Circuit depth:", tiny_isa_optimized.depth())
print("Gate counts:", tiny_isa_optimized.count_ops())
print("Two-qubit gates:", tiny_isa_optimized.num_nonlocal_gates())

NameError: name 'tiny_ansatz' is not defined

In [18]:
import numpy as np

initial_tiny_parameters = np.zeros(
    tiny_isa_optimized.num_parameters
)

print("Submitting 4-qubit chemistry measurement...")

tiny_job = estimator.run([
    (
        tiny_isa_optimized,
        tiny_hamiltonian_optimized,
        initial_tiny_parameters
    )
])

print("Job submitted!")
print("Job ID:", tiny_job.job_id())

NameError: name 'tiny_isa_optimized' is not defined

In [17]:
print(tiny_job.status())

NameError: name 'tiny_job' is not defined

In [38]:
tiny_result = tiny_job.result()

hardware_energy = float(tiny_result[0].data.evs)

print("IBM hardware electronic energy:", hardware_energy, "Hartree")

IBM hardware electronic energy: -1.4827238587730933 Hartree


In [39]:
from qiskit_algorithms import NumPyMinimumEigensolver

exact_solver = NumPyMinimumEigensolver()

exact_result = exact_solver.compute_minimum_eigenvalue(
    tiny_hamiltonian
)

classical_energy = float(exact_result.eigenvalue.real)

print("Classical exact electronic energy:", classical_energy, "Hartree")
print("IBM hardware electronic energy:", hardware_energy, "Hartree")
print("Difference:", abs(hardware_energy - classical_energy), "Hartree")


Classical exact electronic energy: -1.6638934269130738 Hartree
IBM hardware electronic energy: -1.4827238587730933 Hartree
Difference: 0.18116956813998053 Hartree


In [40]:
print("Hamiltonian constants:")
print(tiny_problem.hamiltonian.constants)

energy_offset = sum(
    float(value)
    for value in tiny_problem.hamiltonian.constants.values()
)

print("\nTotal energy offset:", energy_offset, "Hartree")

print(
    "IBM total energy:",
    hardware_energy + energy_offset,
    "Hartree"
)

print(
    "Classical total energy:",
    classical_energy + energy_offset,
    "Hartree"
)

Hamiltonian constants:
{'nuclear_repulsion_energy': np.float64(9.193913160623385), 'ActiveSpaceTransformer': np.float64(-82.49421502570773)}

Total energy offset: -73.30030186508435 Hartree
IBM total energy: -74.78302572385745 Hartree
Classical total energy: -74.96419529199743 Hartree
